> [!IMPORTANT]
> **Disclaimer**: The content and views presented during this session are the author's own and not of any organizations they are associated with or employed at. The code shown in this repository is for illustration and educational purposes only. It is not production-grade; error handling, security, and scalability are not fully addressed.

# Lab 2: Distributed Tracing and Tool Calls using Arize Phoenix
**Faculty Development Programme on Observability for AI Agents**

> [!NOTE]
> **Curriculum Cross-Reference**: This laboratory exercise implements the practical aspects of the **Distributed Tracing and OpenTelemetry** slides detailed in **Section 3 of [00_curriculum_and_agenda.ipynb](00_curriculum_and_agenda.ipynb)**.

### Overview
Autonomous AI agents execute complex, multi-turn reasoning loops (where the agent iterates through multiple thoughts and actions). An agent typically queries a language model, parses the model's reasoning, decides to invoke an external tool (e.g., executing a calculator task or performing a database query), processes the tool's output, and feeds the updated execution context back to the model. 

Traditional single-line logging formats are insufficient to record these hierarchical, parent-child execution boundaries (mapping nested sub-tasks back to the main user request that triggered them). Comprehensive visibility requires **Distributed Tracing** (a method to track request lifecycles across multiple functions, containers, or remote microservices) framework integrations.

### Learning Objectives:
1. Understand how **Spans** (units of work capturing timestamps, metadata, and execution status) and **Traces** (the overall request pathway composed of multiple spans) capture distributed execution hierarchies.
2. Launch **Arize Phoenix** locally to serve as an interactive OpenTelemetry collector backend.
3. Configure the OpenTelemetry SDK to export trace spans.
4. Instrument a ReAct Agent reasoning loop to associate tool calls with parent spans.
5. Visualize the agent's sequential reasoning, model inputs/outputs, and tool executions.

## 1. Launching Arize Phoenix (or Alternative OTel Backends like Jaeger)

Arize Phoenix is an open-source tool that can execute directly within Jupyter Notebooks or run as a standalone local server process (via terminal commands or Docker containers). Utilizing **OpenTelemetry (OTel)** ensures that the agent instrumentation codebase remains **vendor-neutral**.

### 📜 The Evolution of Telemetry Standards: OpenTracing vs. OpenCensus vs. OpenTelemetry
Historically, telemetry instrumentation was split across two standards, causing integration friction:
1. **OpenTracing**: 
   * *Scope*: Focused **exclusively on Traces**.
   * *Implementation*: Provided **only an API specification** (an abstract interface). It did not include SDK libraries, requiring developers to import vendor-specific implementations (e.g., Jaeger client libraries) to generate and ship spans.
2. **OpenCensus**: 
   * *Scope*: Provided coverage for **both Traces and Metrics**.
   * *Implementation*: Open-sourced by Google, it provided **both the API specification and concrete SDK libraries**. Developers could write instrumentation and export data directly to multiple backends using built-in exporters, without vendor-specific libraries.
3. **OpenTelemetry**: Created by merging the OpenTracing and OpenCensus initiatives under the CNCF. OTel provides a single, unified framework defining APIs, SDKs, and collectors to capture the three core telemetry types: **Traces, Metrics, and Logs**.

### 🚀 Spawning a Standalone Arize Phoenix Instance Locally
Although this notebook launches Phoenix in-process using Python APIs (`px.launch_app()`), the collector can be executed as a standalone background service using CLI or Docker commands:

1. **Option A: Python CLI (Virtual Environment)**:
   ```bash
   # Activate the virtual environment and launch the server daemon
   source venv/bin/activate
   phoenix serve
   ```
   *By default, this spawns the server on `http://localhost:6006` and starts listening for OTLP traces.*

2. **Option B: Docker Container**:
   ```bash
   # Pull and execute the official Arize Phoenix image, mapping the UI and OTLP ports
   docker run -d -p 6006:6006 -p 4317:4317 -p 4318:4318 arizeai/phoenix:latest
   ```
   *This exposes the telemetry web analyzer at port 6006, the OTLP/gRPC receiver at port 4317, and the OTLP/HTTP receiver at port 4318.*

Modifying the OTLP exporter endpoint redirects the same trace dataset to alternative open-source or commercial telemetry backends:
1. **Arize Phoenix**: Runs in-notebook at `http://localhost:6006/v1/traces`.
2. **Jaeger**: Industry-standard open-source distributed tracing dashboard. Runs via docker-compose (located in the `docker/` directory) and accepts OTLP traces at `http://localhost:4318/v1/traces`.
3. **Langfuse**: Popular open-source LLM engineering platform. Can be run locally via Docker and supports OTel ingestion.

The tracing server is initiated. By default, Arize Phoenix is launched within the notebook interface. Comments are included to demonstrate routing traces to Jaeger.

In [ ]:
import sys
import os
import re
import time
import socket
sys.path.append(os.path.abspath('..'))

import phoenix as px
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from src.mock_llm import MockLLMClient, MockLLMResponse

# Choose tracing backend: "phoenix" or "jaeger"
TRACING_BACKEND = "phoenix"

def is_port_in_use(port: int) -> bool:
    """
    Checks if a local network port is currently occupied by another process.
    """
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

if TRACING_BACKEND == "phoenix":
    # Production Tip: Set alternative gRPC ports via environment variables to avoid conflicts
    # if another collector (like Jaeger or OTel Collector) is already using default port 4317.
    os.environ["PHOENIX_GRPC_PORT"] = "4325"

    # Check if a local server is already running on web UI port 6006
    if is_port_in_use(6006):
        print("🌍 Standalone Arize Phoenix server detected on port 6006. Reusing active session.")
    else:
        print("Launching local Arize Phoenix trace collector server...")
        try:
            session = px.launch_app()
        except Exception as e:
            print(f"⚠️ Could not launch Phoenix: {e}")

    # Defines the local Arize Phoenix collector HTTP endpoint for tracing ingestion
    exporter_endpoint = "http://localhost:6006/v1/traces"
elif TRACING_BACKEND == "jaeger":
    # Verify that the local docker-compose stack is active:
    # cd ../docker && docker-compose up -d
    # Defines the local containerized Jaeger collector OTLP/HTTP endpoint
    exporter_endpoint = "http://localhost:4318/v1/traces"
    print("Routing traces to local Jaeger collector...")

# Creates the central TracerProvider that manages the trace generation workflow
provider = TracerProvider()
# Sets this provider instance as the global tracer manager for the application
trace.set_tracer_provider(provider)

# Initializes the OTLP exporter pointing to our selected collector endpoint URL
otlp_exporter = OTLPSpanExporter(endpoint=exporter_endpoint)
# Hooks the exporter to the provider using a Simple Processor that publishes spans synchronously
provider.add_span_processor(SimpleSpanProcessor(otlp_exporter))

# Acquires a named tracer instance to generate tracing spans throughout the agent runtime
tracer = trace.get_tracer("agent_observability")
print(f"Telemetry tracing configured. Sending spans to: {exporter_endpoint}")

Overriding of current TracerProvider is not allowed


🌍 Standalone Arize Phoenix server detected on port 6006. Reusing active session.
Telemetry tracing configured. Sending spans to: http://localhost:6006/v1/traces


## 2. Defining Agent Tools with OpenTelemetry Instrumentation

Two tools are defined: a mock **Web Search Tool** and a mock **Calculator Tool**.
Each tool is wrapped within an **OpenTelemetry span** to capture its execution duration, inputs, outputs, and errors automatically.

> [!WARNING]
> **Production Security Note: Shadow APIs and Egress Remediation Across Topologies**
> * **The Threat**: In this laboratory exercise, the agent invokes custom Python tools like `web_search(query)`. In production deployments, these tools make remote network calls. Since tool arguments are generated dynamically by the LLM, a poisoned prompt or injection attack can hijack the query parameters, forcing the tool to make calls to unauthorized external endpoints (known as **Shadow APIs**).
> * **Leading Remediation Practices (By Deployment Topology)**:
>   To enforce egress restriction rules, security teams apply controls tailored to the specific execution platform:
>   1. **Local and Bare-Metal**: Configure local operating system firewalls (e.g., `ufw` or `iptables`) to block outbound sockets and restrict loopback routes.
>   2. **Virtual Machines (VMs)**: Apply Cloud Virtual Private Cloud (VPC) Security Groups or Security Lists to restrict VM outbound routing.
>   3. **PaaS, Serverless, and FaaS (e.g., AWS Lambda, GCP Cloud Run)**: Configure cloud-native VPC Egress firewalls, VPC Service Controls, and route outbound internet traffic exclusively through an enterprise NAT gateway.
>   4. **Kubernetes (K8s)**: Apply **NetworkPolicies** to restrict pod egress traffic to only explicitly permitted destination namespaces and external domain IPs.
>   5. **Unified Gateway Control (All Platforms)**: Regardless of the infrastructure layer, all outbound HTTP/gRPC requests from the agent runtime must route through an authenticated **Secure Egress Forward Proxy** (e.g., Envoy or Squid) configured with strict domain allowlists.

In [5]:
def web_search(query: str) -> str:
    """
    Simulates a web search operation and logs tool execution spans.

    Args:
        query (str): The search phrase or topic.

    Returns:
        str: The retrieved document snippet or search result.
    """
    # A child span is created to isolate search tool execution within the trace tree
    with tracer.start_as_current_span("tool_search") as span:
        span.set_attribute("tool.name", "web_search")
        span.set_attribute("tool.input", query)

        print(f"-> [Tool Search] Searching for: '{query}'")
        time.sleep(0.5) # Simulated lookup latency

        query_clean: str = query.lower()
        if "population of france" in query_clean:
            result: str = "The population of France is estimated to be around 68 million."
        else:
            result = f"Search returned no clear results for '{query}'."

        span.set_attribute("tool.output", result)
        return result

def calculator(expression: str) -> str:
    """
    Evaluates basic arithmetic expressions and handles trace exception recording.

    Args:
        expression (str): String mathematical expression (e.g. '68000000 * 2').

    Returns:
        str: String representation of the numerical result or error details.
    """
    # A child span is created to isolate calculator tool execution
    with tracer.start_as_current_span("tool_calculator") as span:
        span.set_attribute("tool.name", "calculator")
        span.set_attribute("tool.input", expression)

        print(f"-> [Tool Calculator] Evaluating: '{expression}'")
        time.sleep(0.3) # Simulated evaluation latency

        # Sanitization: Non-mathematical characters are removed to protect the execution environment
        expression_clean: str = re.sub(r'[^0-9\+\-\*\/\s]', '', expression)
        try:
            result: str = str(eval(expression_clean))
        except Exception as e:
            result = f"Error evaluating expression: {e}"
            span.record_exception(e)
            span.set_status(trace.StatusCode.ERROR, str(e))

        span.set_attribute("tool.output", result)
        return result

## 3. Building the ReAct Reasoning Loop with Tracing

The ReAct agent runner is constructed.
The overall execution is wrapped in a parent span (`agent_run`), and the reasoning loop is tracked sequentially.
Each LLM invocation initiates a child `llm_call` span.
When a tool is executed, it starts its own span, which automatically nests under the active span context.

### 🤖 Model Generation Provider: MockLLMClient
To implement the agent reasoning loops offline, this lab leverages the local **`MockLLMClient`** (introduced in detail in **[Lab 1](01_structured_logging_cost_carbon.ipynb)**). The client simulates model processing latency, token counts, and costs, allowing us to track OpenTelemetry spans and context boundaries without hitting rate limits or cloud service fees.

In [ ]:
# The mock LLM client is initialized to simulate local inferences without network API charges
llm_client: MockLLMClient = MockLLMClient()

def run_react_agent(user_query: str) -> str:
    """
    Executes a ReAct (Reasoning and Action) loop to answer user queries.
    Establishes parent-child OpenTelemetry span relationships for LLM and tool calls.

    Args:
        user_query (str): The search or mathematical calculation query.

    Returns:
        str: The final generated answer or error message.
    """
    # Create the parent span representing the overall execution lifecycle of this agent request.
    # The 'with' context manager ensures that the span context is correctly closed even if an error is raised.
    with tracer.start_as_current_span("agent_run") as parent_span:
        # Attach the user query as a key-value attribute to the parent span to allow indexing and searching in the UI.
        parent_span.set_attribute("agent.query", user_query)
        print(f"Starting Agent Run for query: '{user_query}'\n")

        # Initialize context history buffer to maintain dialogue loops
        history = [f"User query: {user_query}"]
        max_iterations: int = 5
        iteration: int = 0

        while iteration < max_iterations:
            iteration += 1
            print(f"--- Iteration {iteration} ---")

            # Combine conversation history into a prompt template for the model reasoning step
            prompt: str = "\n".join(history) + "\nChoose your next action (search or calculator or Final Answer)."

            # Start a child span to track the language model inference step.
            # Because this context manager runs inside the 'agent_run' span, OpenTelemetry
            # automatically establishes a parent-child relationship in the trace tree.
            with tracer.start_as_current_span("llm_call") as llm_span:
                # Log the input prompt string as a semantic trace attribute
                llm_span.set_attribute("llm.prompt", prompt)

                # Execute simulated inference using the local mock client
                response: MockLLMResponse = llm_client.generate(prompt)

                # Attach token count and cost metrics as numeric attributes to the child span
                # for cost audit dashboards and system analysis in Arize Phoenix.
                llm_span.set_attribute("llm.response", response.text)
                llm_span.set_attribute("llm.prompt_tokens", response.prompt_tokens)
                llm_span.set_attribute("llm.completion_tokens", response.completion_tokens)
                llm_span.set_attribute("llm.cost", response.cost)

            # Append the model output to dialogue history to maintain context
            history.append(response.text)
            print(f"LLM Output:\n{response.text}\n")

            # Parse the model output to extract actions (tool call or final answer)
            action_match = re.search(r'Action:\s*(\w+)\((.+)\)', response.text)
            final_answer_match = re.search(r'Final Answer:\s*(.+)', response.text)

            # Case A: Model outputs the final answer, terminating the loop
            if final_answer_match:
                final_answer: str = final_answer_match.group(1)
                # Attach final output response attributes to the root span
                parent_span.set_attribute("agent.final_answer", final_answer)
                print("Agent finished successfully!")
                return final_answer

            # Case B: Model decides to call an external tool
            elif action_match:
                tool_name: str = action_match.group(1)
                tool_arg: str = action_match.group(2).strip('"\'')

                # Execute the corresponding tool function.
                # Note: These tool functions ('web_search', 'calculator') are also instrumented
                # with OTel context managers, making them sub-spans nested under 'agent_run'.
                if tool_name == "search":
                    tool_result: str = web_search(tool_arg)
                elif tool_name == "calculator":
                    tool_result = calculator(tool_arg)
                else:
                    tool_result = f"Error: Unknown tool '{tool_name}'"

                # Append tool execution outputs to history buffer
                history.append(f"Tool {tool_name} returned: {tool_result}")
                print(f"Tool Result: {tool_result}\n")

            # Case C: The model outputs malformed instructions that cannot be parsed
            else:
                error_msg: str = "Error: LLM output did not specify an Action or Final Answer."
                # Record exception traces directly to the parent span to alert operators of system failures
                parent_span.record_exception(RuntimeError(error_msg))
                # Update the trace status flag to ERROR to highlight this run in red on dashboards
                parent_span.set_status(trace.StatusCode.ERROR, error_msg)
                return error_msg

        # Case D: The agent loop runs out of steps without reaching a final answer (infinite loop prevention)
        timeout_msg: str = "Agent timed out after maximum iterations."
        parent_span.set_status(trace.StatusCode.ERROR, timeout_msg)
        return timeout_msg

## 4. Running the Agent and Visualizing in Arize Phoenix

The agent is executed with a query that requires both search and math capabilities:
`"What is the population of France multiplied by 2?"`

Following execution, the local Arize Phoenix dashboard is accessed using the session link generated in Section 1 (typically `http://localhost:6006`).
The trace tree visualization includes the parent `agent_run`, alongside nested `llm_call`, `tool_search`, and `tool_calculator` spans.

In [7]:
query = "What is the population of France multiplied by 2?"
result = run_react_agent(query)
print(f"\nFinal Agent Result: {result}")

Starting Agent Run for query: 'What is the population of France multiplied by 2?'

--- Iteration 1 ---
LLM Output:
Thought: I need to find the population of France first.
Action: search("population of France")

-> [Tool Search] Searching for: 'population of France'
Tool Result: The population of France is estimated to be around 68 million.

--- Iteration 2 ---
LLM Output:
Thought: The population of France is 68 million. Now I need to multiply this number by 2.
Action: calculator("68000000 * 2")

-> [Tool Calculator] Evaluating: '68000000 * 2'
Tool Result: 136000000

--- Iteration 3 ---
LLM Output:
Thought: I have the final calculated value of 136,000,000.
Final Answer: The population of France multiplied by 2 is 136,000,000.

Agent finished successfully!

Final Agent Result: The population of France multiplied by 2 is 136,000,000.


### 💡 Tips for Navigating and Interpreting the Arize Phoenix UI
When inspecting the traces in the browser interface (default `http://localhost:6006`), look for the following:

1. **Locating the Trace Run**:
   - On the **Projects** dashboard, select the active workspace.
   - The main trace list displays a list of requests. Look for the trace named **`agent_run`** corresponding to the query: *"What is the population of France multiplied by 2?"*.
   - The list indicates the overall execution status (green for success, red for failures) and total latency.

2. **Analyzing the Nested Trace Tree (Span Hierarchy)**:
   - Click on the `agent_run` trace to open the detailed timeline.
   - Observe the hierarchy of parent-child spans. Notice how child spans (`llm_call`, `tool_search`, and `tool_calculator`) are nested sequentially within the lifetime of the root `agent_run` span.
   - The timeline bars show which step contributed most to the agent's total response latency.

3. **Auditing Inputs and Outputs (Attributes Tab)**:
   - Click on any `llm_call` span in the tree.
   - Under the **Attributes** tab in the right-hand panel, inspect the metadata:
     * `llm.prompt`: Displays the exact text prompt sent to the LLM (including accumulated context history).
     * `llm.response`: Displays the generated thought and action string.
     * `llm.cost`: Displays the financial cost calculated for that specific transaction.
   - Click on `tool_search` or `tool_calculator` to verify `tool.input` and `tool.output` values.

4. **Debugging Exceptions (Red Fails)**:
   - In production runs, if a tool raises a python exception, the failed span is colored red.
   - Clicking on a failed span and selecting the **Exceptions** tab displays the full traceback for diagnostic debugging.

## 5. Production Guidelines and Leading Practices

When deploying distributed tracing for AI agents, the following guidelines ensure scalability and performance:

### 🔌 OpenTelemetry Collector Architecture
* **Asynchronous Offloading**: Applications should not export trace spans directly to remote SaaS backends over the public internet, as this introduces network latency. Instead, run an **OpenTelemetry Collector** as a local sidecar container or host daemon. The application exports spans to the local collector over gRPC or HTTP (low latency), and the collector buffers and exports them asynchronously to the tracing backend.

### 📊 Trace Sampling Strategies
* **Volume Management**: Tracing 100% of agent executions generates massive database storage costs. Production deployments leverage:
  * **Head-Based Sampling (Probability)**: Configuring the OTel SDK to randomly sample a fixed percentage of traces (e.g., 5%).
  * **Tail-Based Sampling**: The collector inspects completed traces and drops normal runs while preserving 100% of traces containing errors, exceptions, or high latency.

### 🛡️ Span Payload Limits
* **Attribute Truncation**: AI agent prompts and outputs can span thousands of tokens. Restrict the maximum attribute length in the OTel SDK configuration to truncate large payloads, preventing memory exhaustion or exporter timeouts in the OTel pipeline.

### Summary of Lab 2
1. **Trace Trees**: Parent-child relationships are constructed where tools and LLM generations exist within a single `agent_run` context.
2. **Standard OTel SDK**: Instrumentation utilizes the official, standardized OpenTelemetry library. This prevents vendor lock-in; the same code can export traces to Jaeger, Zipkin, Phoenix, Datadog, or Honeycomb.
3. **Local Collector**: Arize Phoenix acts as a local OpenTelemetry collector, providing an instant, graphical view of agent execution loops.
4. **Alternative Visualizations**: Using the local `docker/` configuration, Jaeger can be executed and traces routed to it by changing the exporter endpoint to `http://localhost:4318/v1/traces`.

### 📊 Clarifying the Telemetry Landscape: Traces vs. Metrics vs. Logs
To ensure clarity on the different pillars of observability implemented across the labs, the following breakdown maps each type to its generating library and destination backend:

| Lab / Telemetry Type | Generating Library / SDK | Primary Ingestion and Visualization Backend |
| :--- | :--- | :--- |
| **Lab 1: Structured Logs** | Python's `structlog` library | Printed to standard output (`stdout`) for ingestion by shipper daemons (e.g., Vector or FluentBit). |
| **Lab 2: Distributed Traces** | OpenTelemetry (OTel) Tracing SDK | Exported via OTLP to **Arize Phoenix** (acting as the traces collector and visualization UI). |
| **Lab 3: Metrics** | OpenTelemetry (OTel) Metrics SDK | Collected in-memory (and typically routed to Prometheus or Grafana in production). |

The subsequent lab covers **Agent Metrics and Evaluations** to score agent behavior and check for issues such as hallucinations.